<a href="https://colab.research.google.com/github/kalapak-team/fitandsleek_training_colab/blob/main/fitandsleek_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fitandsleek LoRA Training (Colab)

1. **Runtime → Change runtime type → GPU (T4) → Save**
2. Run all cells in order (Runtime → Run all)
3. Download `models/fitandsleek-lora` when finished

Repo: https://github.com/kalapak-team/fitandsleek_training_colab

## 1) Check GPU

In [1]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Go to Runtime → Change runtime type → GPU (T4)')

cuda: True
gpu: Tesla T4


## 2) Clone GitHub repo

In [2]:
import os
REPO = 'fitandsleek_training_colab'
URL = 'https://github.com/kalapak-team/fitandsleek_training_colab.git'

if os.path.isdir(REPO):
    %cd {REPO}
    !git pull
else:
    !git clone {URL}
    %cd {REPO}

!ls -la
!ls data scripts

Cloning into 'fitandsleek_training_colab'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 22 (delta 2), reused 21 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 21.86 KiB | 355.00 KiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/fitandsleek_training_colab
total 96
drwxr-xr-x 6 root root  4096 Jul 31 02:55 .
drwxr-xr-x 1 root root  4096 Jul 31 02:55 ..
-rw-r--r-- 1 root root 29707 Jul 31 02:55 app.py
drwxr-xr-x 2 root root  4096 Jul 31 02:55 data
-rw-r--r-- 1 root root   518 Jul 31 02:55 .env.example
-rw-r--r-- 1 root root  3952 Jul 31 02:55 fitandsleek_training.ipynb
drwxr-xr-x 8 root root  4096 Jul 31 02:55 .git
-rw-r--r-- 1 root root   213 Jul 31 02:55 .gitignore
drwxr-xr-x 2 root root  4096 Jul 31 02:55 models
-rw-r--r-- 1 root root  3193 Jul 31 02:55 products.json
-rw-r--r-- 1 root root   483 Jul 31 02:55 README.md
-rw-r--r-- 1 root root    67 Jul 31 0

## 3) Install training packages

In [3]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.0 MB/s eta 0:00:00


## 4) (Optional) Refresh dataset from store_info.json

In [4]:
!python scripts/prepare_train_data.py
!wc -l data/fitandsleek_train.jsonl

Wrote 365 examples -> /content/fitandsleek_training_colab/data/fitandsleek_train.jsonl
365 data/fitandsleek_train.jsonl


## 5) Train LoRA
This can take ~20–60+ minutes on T4.

In [7]:
!python scripts/train_lora.py

Map: 100% 365/365 [00:00<00:00, 7239.42 examples/s]
Loading weights: 100% 434/434 [00:11<00:00, 36.57it/s]
Tokenizing train dataset: 100% 365/365 [00:00<00:00, 1537.00 examples/s]
Building labels for train dataset: 100% 365/365 [00:00<00:00, 4629.15 examples/s]
Truncating train dataset: 100% 365/365 [00:00<00:00, 4379.30 examples/s]
Dropping fully masked examples from train dataset: 100% 365/365 [00:00<00:00, 6252.23 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
  0% 0/138 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/content/fitandsleek_training_colab/scripts/train_lora.py", line 115, in <module>
    main()
  File "/content/fitandsleek_training_colab/scripts/train_lora.py", line 108, in main
    trainer.train()
  File "/usr/

## 6) Zip adapter for download

In [6]:
import os
from google.colab import files

adapter = 'models/fitandsleek-lora'
if not os.path.isdir(adapter):
    raise SystemExit('Adapter folder not found. Did training finish?')

!zip -r fitandsleek-lora.zip models/fitandsleek-lora
files.download('fitandsleek-lora.zip')
print('Download started: fitandsleek-lora.zip')
print('On PC: unzip into models/fitandsleek-lora then set LORA_ADAPTER=models/fitandsleek-lora in .env')

  adding: models/fitandsleek-lora/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started: fitandsleek-lora.zip
On PC: unzip into models/fitandsleek-lora then set LORA_ADAPTER=models/fitandsleek-lora in .env
